# Imports

In [0]:
# init

from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, window

# Read tables from silver_schema

In [0]:
df_crm_prd_info = spark.table("workspace.silver_schema.crm_prd_info")
display(df_crm_prd_info)
df_crm_prd_info.createOrReplaceTempView("product")

"""
    product has historical data but we only need recent data, that means where product_end_date IS NULL
"""

In [0]:
df_erp_px_cat_g1v2 = spark.table("workspace.silver_schema.erp_px_cat_g1v2")
display(df_erp_px_cat_g1v2)
df_erp_px_cat_g1v2.createOrReplaceTempView("category")

# Apply Business Rule for Gold Layer

In [0]:
# Joining both tables with all columns inclusive

query = """
    SELECT 
        p.*, 
        c.*
    FROM product p
    LEFT JOIN category c
        USING (category_id)
"""

spark.sql(query).display()

In [0]:
# Include only useful tables, retain only current products where product_end_date is null

query = """
   SELECT 
            ROW_NUMBER() OVER(ORDER BY p.product_start_date ASC,p.product_key ASC) AS product_key,
            p.product_id, 
            p.product_key AS product_number, 
            p.product_name, 
            p.category_id,
            c.category,
            c.sub_category,
            c.maintenance,
            p.product_cost AS cost, 
            p.product_line, 
            p.product_start_date AS start_date
            -- p.product_end_date, /* Commented Column cause not needed (All null according to the WHERE) */

           
            

        FROM product p
        LEFT JOIN category c
            USING (category_id)
        
        WHERE product_end_date IS NULL
    
"""

df_dim_product = spark.sql(query)
df_dim_product.display()

In [0]:
# Data quality checks - check for duplicate products
df_dim_product.createOrReplaceTempView("dim_product")
query = f"""

    SELECT 
        product_number, COUNT(*) 
    FROM dim_product  
    GROUP BY product_number
    HAVING COUNT(*) > 1
    
"""

spark.sql(query).display()

"""
    No duplicates found
"""

# Write to gold_schema

In [0]:
df_dim_product.write.mode('overwrite').format('delta').saveAsTable("workspace.gold_schema.dim_product")

# Read from gold_schema

In [0]:
df_gold_dim_product = spark.table("workspace.gold_schema.dim_product")
df_gold_dim_product.display()